# Capstone Two — Data Wrangling
**Project:** Predicting Residential Solar Adoption Across U.S. Counties
**Author:** Colin Segura — Springboard Data Science Career Track

This notebook covers Step 2 of the capstone (data wrangling) and follows the four
substeps from the Data Science Method: **Data Collection, Data Organization, Data
Definition, and Data Cleaning**.

The end product is a single county-level dataframe combining:

- **LBNL Tracking the Sun** — installation-level residential PV data, aggregated up to county-level adoption counts (the modeling target)
- **U.S. Census ACS** — county-level housing, income, and population (denominator and key features)
- **EIA Open Data** — state-level residential retail electricity prices (key feature)

NREL irradiance and DSIRE policy indicators are deferred to the feature-engineering step.

> **Note on data files.** This notebook reads from CSVs in `../data/raw/`. In the
> live project these are downloaded from the source URLs listed in the project
> proposal. The synthetic samples used here are schema-faithful to the real
> sources, so the wrangling code below runs unchanged against the real downloads.


## 1. Data Collection

Goal: load each raw dataset into a pandas DataFrame. The three sources arrive
in completely different shapes — installation-level, county-level wide, and
state-year long — so each needs its own load step before they can be joined.


In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Quick look at the raw directory
for f in sorted(glob.glob(str(RAW_DIR / "*.csv"))):
    size_kb = os.path.getsize(f) / 1024
    print(f"{Path(f).name:42s}  {size_kb:8.1f} KB")


### 1.1 Load Tracking the Sun (target source)

A note on FIPS codes: county FIPS are 5-digit strings, but several states have
codes that begin with `0` (California `06xxx`, Colorado `08xxx`, Connecticut
`09xxx`, etc). If pandas reads them as integers the leading zero gets dropped
and joins to ACS silently fail. Force the column to string on read.


In [2]:
tts = pd.read_csv(
    RAW_DIR / "tracking_the_sun_sample.csv",
    dtype={"county_fips": "string", "zip_code": "string"},
)
print(f"Tracking the Sun: {tts.shape[0]:,} rows, {tts.shape[1]} columns")
tts.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/tracking_the_sun_sample.csv'

In [ ]:
print(tts.head().to_string())

### 1.2 Load ACS county data

ACS comes with a `GEO_ID` like `0500000US06037`. The last 5 digits are the
county FIPS, which is what we need for joining.


In [ ]:
acs = pd.read_csv(RAW_DIR / "acs_county_housing_income.csv")
print(f"ACS county data: {acs.shape[0]:,} rows, {acs.shape[1]} columns")
print(acs.head().to_string())


### 1.3 Load EIA state-level electricity prices

EIA's API returns long-format state-year-sector rows. We filter to the
residential sector and join later on state code.


In [ ]:
eia = pd.read_csv(RAW_DIR / "eia_state_electricity_prices.csv")
print(f"EIA state prices: {eia.shape[0]:,} rows, {eia.shape[1]} columns")
print(eia.head().to_string())


## 2. Data Organization

Goal: confirm the project's file structure and review what we have on disk.

```
capstone_two/
├── data/
│   ├── raw/          <- original downloads, never edited
│   └── processed/    <- cleaned outputs from this notebook
├── notebooks/
│   └── 01_data_wrangling.ipynb
└── scripts/
```

This convention (`raw` is read-only, `processed` is generated by code)
is the standard data-science project layout. It guarantees that any
intermediate file can be regenerated from the raw inputs and the
notebook, which is what makes the project reproducible.


In [ ]:
# Sanity check: list files using glob (per the assignment hint)
print("Raw inputs:")
for f in sorted(glob.glob(str(RAW_DIR / "*"))):
    print(" ", Path(f).name)

print("\nProcessed outputs:")
processed_files = sorted(glob.glob(str(PROCESSED_DIR / "*")))
print(" ", processed_files if processed_files else "(none yet)")


## 3. Data Definition

Goal: understand each dataset's columns, types, and ranges before cleaning.


### 3.1 Tracking the Sun

In [ ]:
print("Columns and dtypes:")
print(tts.dtypes)
print(f"\nShape: {tts.shape}")


In [ ]:
print("Tracking the Sun — summary statistics for numeric columns:")
print(tts.describe(include=[np.number]).to_string())


In [ ]:
print("Tracking the Sun — unique value counts for categorical columns:")
for col in ["data_provider_1", "customer_segment", "state"]:
    print(f"\n  {col} ({tts[col].nunique()} unique values):")
    print(tts[col].value_counts(dropna=False).head(10).to_string())


### 3.2 ACS county data

The ACS column codes are not human-readable. Rename now so ACS code is
self-documenting.

| ACS code | Meaning |
|---|---|
| B19013_001E | Median household income |
| B25001_001E | Total housing units |
| B25003_002E | Owner-occupied housing units |
| B25035_001E | Median year structure built |
| B01003_001E | Total population |


In [ ]:
acs = acs.rename(columns={
    "B19013_001E": "median_household_income",
    "B25001_001E": "total_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25035_001E": "median_year_built",
    "B01003_001E": "population",
})
print("ACS columns after rename:")
print(acs.columns.tolist())
print(f"\nACS shape: {acs.shape}")
print("\nACS dtypes:")
print(acs.dtypes)


In [ ]:
print("ACS — summary statistics:")
print(acs.describe().to_string())


### 3.3 EIA state prices

In [ ]:
print("EIA columns and dtypes:")
print(eia.dtypes)
print(f"\nEIA shape: {eia.shape}")
print(f"\nUnique states: {eia['stateid'].nunique()}")
print(f"Year range: {eia['period'].min()} to {eia['period'].max()}")
print(f"Sectors present: {eia['sectorid'].unique().tolist()}")


In [ ]:
print("EIA price summary (cents/kWh):")
print(eia['price'].describe().to_string())


## 4. Data Cleaning

This is where the bulk of the wrangling happens. Each subsection below
addresses a specific data issue surfaced during definition.


### 4.1 Tracking the Sun: filter to residential and drop bad rows

The proposal scopes the analysis to **residential** PV (solar panels). The raw data includes
some commercial rows that need to be dropped before aggregation.


In [ ]:
print("Before residential filter:", len(tts))
print(tts['customer_segment'].value_counts(dropna=False).to_string())

tts_res = tts[tts['customer_segment'] == 'RES'].copy()
print("\nAfter residential filter:", len(tts_res))


### 4.2 Tracking the Sun: handle missing FIPS codes

A small number of installations have a missing `county_fips`. These rows can't
be aggregated to county level so they have to be dropped, but the count should
be small. If it isn't, that's a data-quality flag worth investigating.


In [ ]:
missing_fips = tts_res['county_fips'].isna().sum()
pct_missing = 100 * missing_fips / len(tts_res)
print(f"Rows with missing county_fips: {missing_fips:,}  ({pct_missing:.2f}%)")

# Decision: drop rows missing FIPS. <0.5% of records, no way to recover.
tts_res = tts_res.dropna(subset=['county_fips']).copy()
print(f"After dropping: {len(tts_res):,} rows")


### 4.3 Tracking the Sun: handle exact duplicate rows

Real TTS dataset releases occasionally include duplicate installation records from
overlapping data-provider feeds. Drop exact duplicates.


In [ ]:
dupes = tts_res.duplicated().sum()
print(f"Exact duplicate rows: {dupes:,}")

tts_res = tts_res.drop_duplicates().copy()
print(f"After dedup: {len(tts_res):,} rows")


### 4.4 Tracking the Sun: handle outliers in cost and system size

Three outlier patterns to address:

1. **Zero cost** — clearly a data entry error; treat as missing
2. **Absurdly high cost** (>$200,000 for a residential system) — likely a unit error (cents vs dollars); treat as missing
3. **Oversized systems** (>30 kW) — almost certainly commercial misclassified as residential; drop

For cost, we *don't* drop — many rows have legitimate missing cost data and
we'll either impute later or use cost as an optional feature.


In [ ]:
print("System size distribution before cleaning:")
print(tts_res['system_size_DC'].describe().to_string())

# Drop nonsensical sizes: must be > 0 and <= 30 kW (residential ceiling).
# Negative or zero values are data entry errors; >30 kW is almost certainly
# commercial misclassified as residential.
bad_size = ((tts_res['system_size_DC'] <= 0) | (tts_res['system_size_DC'] > 30)).sum()
neg_size = (tts_res['system_size_DC'] <= 0).sum()
oversized = (tts_res['system_size_DC'] > 30).sum()
print(f"\nNonsensical system sizes being dropped: {bad_size}")
print(f"  - zero or negative size: {neg_size}")
print(f"  - oversized (>30 kW):    {oversized}")
tts_res = tts_res[(tts_res['system_size_DC'] > 0) & (tts_res['system_size_DC'] <= 30)].copy()

# For cost: convert zero and absurd values to NaN (don't drop the row)
zero_cost = (tts_res['total_installed_price'] == 0).sum()
absurd_cost = (tts_res['total_installed_price'] > 200000).sum()
print(f"Zero-cost rows being set to NaN: {zero_cost}")
print(f"Absurd-cost (>$200k) rows being set to NaN: {absurd_cost}")

tts_res.loc[tts_res['total_installed_price'] == 0, 'total_installed_price'] = np.nan
tts_res.loc[tts_res['total_installed_price'] > 200000, 'total_installed_price'] = np.nan

# Cost-per-watt sanity column for downstream use
tts_res['cost_per_watt'] = tts_res['total_installed_price'] / (tts_res['system_size_DC'] * 1000)

print("\nSystem size distribution after cleaning:")
print(tts_res['system_size_DC'].describe().to_string())
print("\nCost-per-watt distribution (NaN where cost was missing/bad):")
print(tts_res['cost_per_watt'].describe().to_string())


### 4.5 Tracking the Sun: aggregate to county level

This is the key transformation: from one row per installation to one row per
county, which is the unit of analysis for the project.

Target: `installs_total` (cumulative installations) and `installed_kw_total`
(cumulative direct current capacity). We'll convert these to per-capita rates after
joining ACS in step 4.7.


In [ ]:
tts_county = tts_res.groupby('county_fips').agg(
    installs_total=('system_id_1', 'count'),
    installed_kw_total=('system_size_DC', 'sum'),
    median_system_size_kw=('system_size_DC', 'median'),
    median_cost_per_watt=('cost_per_watt', 'median'),
    state=('state', 'first'),
    county_name=('county', 'first'),
).reset_index()

print(f"County-level TTS dataset: {tts_county.shape}")
print(tts_county.head(8).to_string())


### 4.6 ACS: handle suppressed values (-666666666)

The U.S. Census uses negative sentinel codes (`-666666666`, `-999999999`, etc.)
to indicate suppressed estimates. These need to become real NaNs before any
arithmetic. Reference: https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html


In [ ]:
# Replace any large negative sentinel values with NaN across numeric ACS columns
acs_numeric_cols = ['median_household_income', 'total_housing_units',
                    'owner_occupied_units', 'median_year_built', 'population']

before_na = acs[acs_numeric_cols].isna().sum().sum()
for col in acs_numeric_cols:
    acs.loc[acs[col] < 0, col] = np.nan
after_na = acs[acs_numeric_cols].isna().sum().sum()

print(f"Sentinel values converted to NaN: {after_na - before_na}")
print("\nNaN counts per ACS column:")
print(acs[acs_numeric_cols].isna().sum().to_string())


### 4.7 ACS: extract county FIPS from GEO_ID

The `GEO_ID` field looks like `0500000US06037`. We need just the trailing
5-digit FIPS for the join key.


In [ ]:
acs['county_fips'] = acs['GEO_ID'].str[-5:]
print("Sample of extracted FIPS codes:")
print(acs[['GEO_ID', 'county_fips', 'NAME']].head().to_string())

# Confirm none lost their leading zero
fips_lengths = acs['county_fips'].str.len().value_counts()
print(f"\nFIPS length distribution (should all be 5):\n{fips_lengths.to_string()}")


### 4.8 Merge TTS county aggregates with ACS

This is the central join. We use a **left join from ACS to TTS dataset** because
ACS gives us the universe of counties — including ones with zero
installations, which are valid observations for adoption modeling (a
county with very low solar adoption is still a data point).


In [ ]:
county = acs.merge(
    tts_county,
    on='county_fips',
    how='left',
    suffixes=('_acs', '_tts')
)

# Counties with no installations should have 0, not NaN, for installs_total / installed_kw_total
fill_zero = ['installs_total', 'installed_kw_total']
for col in fill_zero:
    county[col] = county[col].fillna(0)

print(f"Merged county dataset: {county.shape}")
print("\nJoin diagnostic:")
print(f"  Counties in ACS:          {len(acs)}")
print(f"  Counties in TTS aggregate: {len(tts_county)}")
print(f"  Counties in merged:        {len(county)}")
print(f"  Counties with 0 installs:  {(county['installs_total'] == 0).sum()}")


### 4.9 Build the per-capita target variable

The proposal specifies the modeling target as **installations per 1,000
owner-occupied households**. We need owner-occupied count from ACS to
build it.


In [ ]:
# Avoid divide-by-zero / divide-by-NaN
county['installs_per_1k_oo_hh'] = (
    county['installs_total'] / county['owner_occupied_units'] * 1000
)
county['kw_per_1k_oo_hh'] = (
    county['installed_kw_total'] / county['owner_occupied_units'] * 1000
)

print("Target variable distribution (installs per 1,000 owner-occupied HH):")
print(county['installs_per_1k_oo_hh'].describe().to_string())


### 4.10 Merge state-level electricity prices

EIA prices are state-year. For the cross-sectional model we'll use the
**most recent year** (2023). Time-varying versions can be built later.


In [ ]:
# Filter to residential sector and most recent year
eia_recent = eia[(eia['sectorid'] == 'RES') & (eia['period'] == eia['period'].max())].copy()
eia_recent = eia_recent[['stateid', 'price']].rename(
    columns={'stateid': 'state', 'price': 'residential_price_cents_kwh'}
)
print(f"EIA most-recent-year slice: {eia_recent.shape}")
print(eia_recent.head().to_string())


In [ ]:
county = county.merge(eia_recent, on='state', how='left')

# Diagnose any unmatched states
unmatched = county[county['residential_price_cents_kwh'].isna()]
print(f"Counties unmatched on state-price join: {len(unmatched)}")
print(f"\nFinal merged shape: {county.shape}")


### 4.11 Final tidy and save

Drop intermediate columns, reorder for readability, and save to the
processed directory.


In [ ]:
# Drop the raw GEO_ID and ACS code columns we no longer need
drop_cols = ['GEO_ID', 'NAME']
county = county.drop(columns=[c for c in drop_cols if c in county.columns])

# Reorder columns: identifiers, target, then features
preferred_order = [
    'county_fips', 'county_name', 'state',
    # targets
    'installs_total', 'installed_kw_total',
    'installs_per_1k_oo_hh', 'kw_per_1k_oo_hh',
    # features
    'median_household_income', 'owner_occupied_units', 'total_housing_units',
    'median_year_built', 'population',
    'residential_price_cents_kwh',
    'median_system_size_kw', 'median_cost_per_watt',
]
county = county[[c for c in preferred_order if c in county.columns]]

print(f"Final shape: {county.shape}")
print("\nFinal columns:")
print(county.dtypes.to_string())

print("\nFirst 10 rows of cleaned county dataset:")
print(county.head(10).to_string())


In [ ]:
print("Final summary statistics:")
print(county.describe().to_string())

print("\nMissing-value counts:")
print(county.isna().sum().to_string())


In [ ]:
out_path = PROCESSED_DIR / "county_solar_adoption.csv"
county.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")
print(f"  Rows:    {len(county):,}")
print(f"  Columns: {len(county.columns)}")
